# 🚀 Hypersonic Physics Dataset – Sample Visualisation Notebook

**Author**: Dr. Aurthur Vimalachandran – Rocket and Aeronautical Engineer  
**Repository**: https://github.com/aurthur001-oss/Hypersonic-Dataset-  

---

This notebook demonstrates how to load, explore, and visualise the Hypersonic Physics Dataset (138,600 data points).  
The dataset covers flight conditions (Mach 5–25), thermal environments, coolant properties, and material survival analysis.

### Sections
1. Load & Inspect the Dataset  
2. Statistical Summary  
3. Mach Number vs Aerodynamic Heating  
4. Altitude vs Heat Flux  
5. Material Comparison – Wall Temperature  
6. Thermal Regime Distribution  
7. Cooling Efficiency vs Mach Number  
8. Survival Analysis by Material  
9. Correlation Heatmap  

In [ ]:
# ── 0. Install dependencies (run once if needed) ──────────────────────────────
# !pip install pandas matplotlib seaborn numpy

In [ ]:
# ── 1. Load & Inspect the Dataset ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})
PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff', '#56d364']

# ─── Load ────────────────────────────────────────────────────────────────────
DATA_PATH = '../hypersonic_dataset_138600pts.csv'   # adjust if running from root
df = pd.read_csv(DATA_PATH)

print(f'Shape  : {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── 2. Statistical Summary ─────────────────────────────────────────────────────
summary = df.describe().T[['mean','std','min','50%','max']]
summary.columns = ['Mean','Std Dev','Min','Median','Max']
print(summary.to_string())

In [ ]:
# ── 3. Mach Number vs Aerodynamic Heating ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

grouped = df.groupby('Mach')[['q_FayRiddell_Wm2','q_SuttonGraves_Wm2','q_BoundaryLayer_Wm2']].mean()

ax.plot(grouped.index, grouped['q_FayRiddell_Wm2'],   marker='o', color=PALETTE[0], label='Fay-Riddell')
ax.plot(grouped.index, grouped['q_SuttonGraves_Wm2'], marker='s', color=PALETTE[2], label='Sutton-Graves')
ax.plot(grouped.index, grouped['q_BoundaryLayer_Wm2'],marker='^', color=PALETTE[1], label='Boundary Layer')

ax.set_xlabel('Mach Number')
ax.set_ylabel('Heat Flux (W/m²)')
ax.set_title('Aerodynamic Heat Flux vs Mach Number', fontsize=14, color='#e6edf3')
ax.legend()
ax.grid(True)
ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Altitude vs Heat Flux (scatter) ────────────────────────────────────────
sample = df.sample(3000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(
    sample['Altitude_km'], sample['q_FayRiddell_Wm2'],
    c=sample['Mach'], cmap='plasma', alpha=0.6, s=10
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Mach Number', color='#e6edf3')
cbar.ax.yaxis.set_tick_params(color='#8b949e')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='#8b949e')

ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Fay-Riddell Heat Flux (W/m²)')
ax.set_title('Altitude vs Heat Flux — coloured by Mach Number', fontsize=14)
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Material Comparison – Outer Wall Temperature ────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

materials = df['Material'].unique()
data_by_material = [df[df['Material'] == m]['T_wall_outer_K'].dropna().values for m in materials]

bp = ax.boxplot(
    data_by_material,
    patch_artist=True,
    labels=materials,
    medianprops=dict(color='#f78166', linewidth=2)
)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Material')
ax.set_ylabel('Outer Wall Temperature (K)')
ax.set_title('Wall Temperature Distribution by Material', fontsize=14)
ax.grid(True, axis='y')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Thermal Regime Distribution ────────────────────────────────────────────
regime_counts = df['Thermal_Regime'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(regime_counts.index, regime_counts.values,
               color=PALETTE[:len(regime_counts)], edgecolor='none')
for bar, val in zip(bars, regime_counts.values):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', ha='left', fontsize=10, color='#8b949e')

ax.set_xlabel('Number of Data Points')
ax.set_title('Thermal Regime Distribution', fontsize=14)
ax.grid(True, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Cooling Efficiency vs Mach Number ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot – cooling efficiency
mach_vals = sorted(df['Mach'].unique())
cool_data  = [df[df['Mach'] == m]['eta_cooling'].dropna().values for m in mach_vals]
vp = axes[0].violinplot(cool_data, positions=mach_vals, widths=0.6,
                         showmedians=True, showextrema=False)
for body in vp['bodies']:
    body.set_facecolor(PALETTE[0])
    body.set_alpha(0.6)
vp['cmedians'].set_color(PALETTE[2])
axes[0].set_xlabel('Mach Number')
axes[0].set_ylabel('Cooling Efficiency (η)')
axes[0].set_title('Cooling Efficiency Distribution', fontsize=13)
axes[0].grid(True)

# Line – mean Hc (heat load capacity) per Mach
hc_mean = df.groupby('Mach')['Hc'].mean()
axes[1].plot(hc_mean.index, hc_mean.values, color=PALETTE[3], marker='D', linewidth=2)
axes[1].fill_between(hc_mean.index, hc_mean.values, alpha=0.2, color=PALETTE[3])
axes[1].set_xlabel('Mach Number')
axes[1].set_ylabel('Mean Heat Capacity Ratio (Hc)')
axes[1].set_title('Heat Capacity Ratio vs Mach', fontsize=13)
axes[1].grid(True)

plt.suptitle('Cooling System Analysis', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. Survival Analysis by Material ──────────────────────────────────────────
survival = df.groupby('Material')['Survived'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 5))
colors = [PALETTE[1] if v >= 80 else PALETTE[2] for v in survival.values]
bars = ax.bar(survival.index, survival.values, color=colors, edgecolor='none', width=0.55)

for bar, val in zip(bars, survival.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11)

ax.axhline(80, color='#f78166', linestyle='--', linewidth=1.2, label='80% threshold')
ax.set_ylim(0, 110)
ax.set_xlabel('Material')
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Material Survival Rate across all Flight Conditions', fontsize=14)
ax.legend()
ax.grid(True, axis='y')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9. Correlation Heatmap ─────────────────────────────────────────────────────
key_cols = [
    'Mach','Altitude_km','T_inf_K','P_inf_Pa','rho_inf_kgm3','V_inf_ms',
    'T_aw_K','q_FayRiddell_Wm2','q_w_design_Wm2','T_wall_outer_K',
    'eta_cooling','Hc','Survived'
]
corr = df[key_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='#21262d',
    ax=ax,
    annot_kws={'size': 8, 'color': '#e6edf3'}
)
ax.set_title('Feature Correlation Matrix', fontsize=15, pad=14)
plt.xticks(rotation=40, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## Summary

| Section | Key Finding |
|---------|-------------|
| Heat Flux vs Mach | Heat flux rises sharply above Mach 15 |
| Altitude Effect | Higher altitude → lower density → reduced heating |
| Material Temps | ZrB2 and HfC sustain highest wall temperatures |
| Thermal Regime | Majority of cases fall in HIGHLY_SAFE or SAFE regimes |
| Cooling Efficiency | Efficiency generally above 0.9 for Mach < 15 |
| Survival Rate | Carbon-Carbon and ZrB2 show highest survival across conditions |

---
*Verified on 2026-06-25 · Dr. Aurthur Vimalachandran*